<h1><font color="#113D68" size=6>Deep Learning en Python con PyTorch</font></h1>

<h1><font color="#113D68" size=5>Parte 3. MultiLayer Perceptron</font></h1>

<h1><font color="#113D68" size=4>5. Evaluación de modelos</font></h1>

<br><br>
<div style="text-align: right">
<font color="#113D68" size=3>Manuel Castillo Cara</font><br>

</div>

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
More information about [Manuel Castillo-Cara](https://www.manuelcastillo.eu/)

<div class="alert alert-block alert-info">

<i class="fa fa-info-circle" aria-hidden="true"></i>
Puedes ver más cursos de Inteligencia Artificial, Machine Learning y Deep Learning en mi [página web](https://www.manuelcastillo.eu/udemy/)


---

<a id="indice"></a>
<h2><font color="#004D7F" size=5>Índice</font></h2>

* [0. Contexto](#section0)
* [1. División de datos](#section1)
* [2. Entrenamiento de un Modelo PyTorch con Validación](#section2)
* [3. Entrenamiento con Validación](#section3)
* [4. Validación Cruzada k-Fold](#section4)

---
<a id="section0"></a>
# <font color="#004D7F" size=6> 0. Contexto</font>

En este capítulo, descubrirás el flujo de trabajo necesario para evaluar de manera robusta el rendimiento de los modelos. Usamos PyTorch para construir nuestros modelos como ejemplos, pero el método también se puede aplicar a otros modelos. Al completar este capítulo, sabrás:

- Cómo evaluar un modelo de PyTorch utilizando un conjunto de verificación.
- Cómo evaluar un modelo de PyTorch con validación cruzada k-fold.

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
[PyTorch tutorials.](https://pytorch.org/tutorials/)

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
[PyTorch Documentation](https://pytorch.org/docs/stable/index.html).

---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section1"></a>
# <font color="#004D7F" size=6>1. Evaluación empírica de modelos</font>

Al diseñar y configurar un modelo de aprendizaje profundo desde cero, hay muchas decisiones que tomar. Esto incluye decisiones de diseño, como cuántas capas usar en un modelo de aprendizaje profundo, cuán grandes son cada una de las capas y qué tipo de capas o funciones de activación utilizar. También puede incluir la elección de la función de pérdida, el algoritmo de optimización, el número de épocas para entrenar y la interpretación de la salida del modelo. Afortunadamente, a veces puedes copiar la estructura de las redes de otras personas. En otras ocasiones, puedes hacer tu elección utilizando algunas heurísticas.

Para determinar si has tomado una buena decisión o no, la mejor manera es comparar múltiples alternativas evaluándolas empíricamente con datos reales. 

El aprendizaje profundo se utiliza a menudo en problemas que cuentan con conjuntos de datos muy grandes, es decir, decenas de miles o incluso cientos de miles de muestras de datos. Esto proporciona suficientes datos para realizar pruebas. Pero necesitas tener una estrategia de prueba robusta para estimar el rendimiento de tu modelo en datos no vistos. En función de eso, puedes tener una métrica para comparar diferentes configuraciones del modelo.

---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section2"></a>
# <font color="#004D7F" size=6>2. División de datos</font>

- Usar todo el conjunto puede aumentar innecesariamente la complejidad y el tiempo de entrenamiento.

- Más datos no siempre resultan en mejores resultados.

- Con grandes cantidades de datos, se recomienda dividir el conjunto:
  - Una porción para el entrenamiento del modelo.
  - Otra porción como conjunto de test para evaluación.

- Este proceso se conoce como "división de train-test".

- Ejemplo con el conjunto de datos de diabetes de Pima Indians (768 muestras):
  - 66% para entrenamiento
  - 34% para prueba

- Esta técnica asegura una evaluación más robusta del rendimiento del modelo en datos no vistos.

In [1]:
# Descarga de datos (necesario en Colab): se guardan en ./Datasets
import os, urllib.request

os.makedirs('Datasets', exist_ok=True)
urls = {
    'pima-indians-diabetes.csv':
        'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.csv',
}
for nombre, url in urls.items():
    if not os.path.exists(f'Datasets/{nombre}'):
        urllib.request.urlretrieve(url, f'Datasets/{nombre}')

In [2]:
import numpy as np
data = np.loadtxt("Datasets/pima-indians-diabetes.csv", delimiter=",")
# find the boundary at 66% of total samples
count = len(data)
n_train = int(count * 0.66)
# split the data at the boundary
train_data = data[:n_train]
test_data = data[n_train:]

- La elección del 66% para el conjunto de entrenamiento es arbitraria.

- Es importante que los datos estén barajados antes de la división.

- Si los datos originales están ordenados, puede resultar en conjuntos de prueba no representativos.

- Opciones para asegurar una división adecuada:
  - Usar `np.random.shuffle(data)` antes de la división.
  
  - Utilizar `train_test_split()` para la división, que es una práctica común.

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
Más información sobre [`train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html).

In [3]:
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(data, test_size=0.33)

---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section3"></a>
# <font color="#004D7F" size=6>3. Entrenamiento con Validación</font>

- En cada iteración del entrenamiento, se extrae un lote del conjunto de entrenamiento.

- El lote se envía al modelo en el paso hacia adelante.

- Se calcula el gradiente en el paso hacia atrás y se actualizan los pesos.

- Aunque se usa la entropía cruzada binaria como métrica de pérdida, el accuracy puede ser de mayor interés.

- Cálculo del accuracy:
  1. Redondear la salida (0 a 1) al entero más cercano para obtener valores binarios (0 o 1).
  2. Contar cuántas predicciones coinciden con las etiquetas reales.

- La predicción es `y_pred`, que es la salida del modelo para `X_batch`.

Este código permite monitorear tanto la pérdida como el accuracy durante el entrenamiento.

In [4]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import tqdm

X = data[: , 0:8]
y = data[: , 8]
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32).reshape(-1, 1)
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, shuffle=True)

In [5]:
model = nn.Sequential(
    nn.Linear(8, 12),
    nn.ReLU(),
    nn.Linear(12, 8),
    nn.ReLU(),
    nn.Linear(8, 1),
    nn.Sigmoid()
)

# loss function and optimizer
loss_fn = nn.BCELoss()  # binary cross-entropy
optimizer = optim.Adam(model.parameters(), lr=0.0001)

n_epochs = 50    
batch_size = 10  
batches_per_epoch = len(X_train) // batch_size

for epoch in range(n_epochs):
    with tqdm.trange(batches_per_epoch, unit="batch", mininterval=0) as bar:
        bar.set_description(f"Epoch {epoch}")
        for i in bar:
            # take a batch
            start = i * batch_size
            X_batch = X_train[start:start+batch_size]
            y_batch = y_train[start:start+batch_size]
            # forward pass
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            # backward pass
            optimizer.zero_grad()
            loss.backward()
            # update weights
            optimizer.step()
            # print progress
            acc = (y_pred.round() == y_batch).float().mean()
            bar.set_postfix(
                loss=float(loss),
                acc=float(acc)
            )

  0%|          | 0/53 [00:00<?, ?batch/s]

Epoch 0:   0%|          | 0/53 [00:00<?, ?batch/s]

/tmp/ipykernel_439181/1030009450.py:37: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:838.)
  loss=float(loss),
Epoch 0:   0%|          | 0/53 [00:00<?, ?batch/s, acc=0.5, loss=2.09]

Epoch 0:   2%|▏         | 1/53 [00:00<00:00, 323.14batch/s, acc=0.5, loss=2.09]

Epoch 0:   2%|▏         | 1/53 [00:00<00:00, 323.14batch/s, acc=0.5, loss=2.39]

...

Epoch 49:  96%|█████████▌| 51/53 [00:00<00:00, 723.52batch/s, acc=1, loss=0.28]   

Epoch 49:  98%|█████████▊| 52/53 [00:00<00:00, 685.13batch/s, acc=1, loss=0.28]

Epoch 49:  98%|█████████▊| 52/53 [00:00<00:00, 685.13batch/s, acc=1, loss=0.387]

Epoch 49: 100%|██████████| 53/53 [00:00<00:00, 681.72batch/s, acc=1, loss=0.387]

Epoch 49: 100%|██████████| 53/53 [00:00<00:00, 725.30batch/s, acc=1, loss=0.387]

Veamos algunos detalles:

- Usar `X_batch` y `y_batch` para calcular el accuracy puede ser problemático.

- El modelo se ajusta para predecir `y_batch` a partir de `X_batch`.

- Verificar si `y_pred` coincide con `y_batch` puede considerarse como "hacer trampa".

- El modelo podría simplemente memorizar las respuestas en lugar de inferirlas.

- En modelos de aprendizaje profundo complejos, es difícil distinguir entre memorización e inferencia.

- La mejor práctica es calcular el accuracy usando el conjunto de prueba (`X_test`).

Este enfoque proporciona una evaluación más confiable del rendimiento del modelo.

In [6]:
for epoch in range(n_epochs):
    with tqdm.trange(batches_per_epoch, unit="batch", mininterval=0) as bar:
        bar.set_description(f"Epoch {epoch}")
        for i in bar:
            # take a batch
            start = i * batch_size
            X_batch = X_train[start:start+batch_size]
            y_batch = y_train[start:start+batch_size]
            # forward pass
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            # backward pass
            optimizer.zero_grad()
            loss.backward()
            # update weights
            optimizer.step()
            # print progress
            acc = (y_pred.round() == y_batch).float().mean()
            bar.set_postfix(
                loss=float(loss),
                acc=float(acc)
            )
    # evaluate model at end of epoch
    y_pred = model(X_test)
    acc = (y_pred.round() == y_test).float().mean()
    acc = float(acc)
    print(f"End of {epoch}: accuracy en test = {acc*100:.1f}%")

  0%|          | 0/53 [00:00<?, ?batch/s]

Epoch 0:   0%|          | 0/53 [00:00<?, ?batch/s]

Epoch 0:   0%|          | 0/53 [00:00<?, ?batch/s, acc=0.6, loss=0.715]

Epoch 0:   2%|▏         | 1/53 [00:00<00:00, 592.92batch/s, acc=0.6, loss=0.715]

Epoch 0:   2%|▏         | 1/53 [00:00<00:00, 592.92batch/s, acc=0.8, loss=0.485]

...

Epoch 49:  98%|█████████▊| 52/53 [00:00<00:00, 632.40batch/s, acc=1, loss=0.244]

Epoch 49:  98%|█████████▊| 52/53 [00:00<00:00, 632.40batch/s, acc=0.9, loss=0.354]

Epoch 49: 100%|██████████| 53/53 [00:00<00:00, 642.55batch/s, acc=0.9, loss=0.354]

Epoch 49: 100%|██████████| 53/53 [00:00<00:00, 637.83batch/s, acc=0.9, loss=0.354]

End of 49: accuracy en test = 68.0%


- La `acc` en el bucle interno es una métrica de progreso, similar a la métrica de pérdida.

- Se espera que el accuracy mejore a medida que la métrica de pérdida disminuye.

- Al final de cada época, se calcula el accuracy usando `X_test`.

- El proceso para calcular el accuracy de test:

  1. Pasar el conjunto de test al modelo.

  2. Obtener las predicciones.

  3. Contar el número de coincidencias con las etiquetas reales.

- El accuracy de test es la métrica más importante a monitorear.

- Signos de alerta durante el entrenamiento:

  - El accuracy de test no mejora.

  - El accuracy de test disminuye.

- Si se observan estos signos, se debe considerar interrumpir el entrenamiento para evitar sobreajuste.

- Indicador de sobreajuste: 

  - El accuracy del conjunto de entrenamiento aumenta.
  
  - El accuracy del conjunto de test disminuye.

Este enfoque proporciona una evaluación más robusta y confiable del rendimiento del modelo.

<a id="section4"></a>
# <font color="#004D7F" size=6>4. Validación Cruzada k-Fold</font>

Para una comparación más robusta, se recomienda usar validación cruzada k-fold.

- Validación cruzada k-fold:
  - Repite el proceso de entrenamiento k veces.

  - Cada vez usa una composición diferente de conjuntos de entrenamiento y test.

  - Resulta en k modelos y k puntuaciones de accuracy.

- Se considera tanto el accuracy promedio como la desviación estándar.

- La desviación estándar indica la consistencia de las puntuaciones de accuracy.

- Implementación de validación cruzada k-fold:
  1. Envolver el bucle de entrenamiento en una función.

  2. Repetir el entrenamiento k veces con diferentes divisiones de datos.

  3. Calcular el accuracy promedio y la desviación estándar.

Lo primero es llevar todo a una función

In [7]:
def model_train(X_train, y_train, X_test, y_test):

    model = nn.Sequential(
        nn.Linear(8, 12),
        nn.ReLU(),
        nn.Linear(12, 8),
        nn.ReLU(),
        nn.Linear(8, 1),
        nn.Sigmoid()
    )

    loss_fn = nn.BCELoss()  
    optimizer = optim.Adam(model.parameters(), lr=0.0001)

    n_epochs = 25   
    batch_size = 10 
    batches_per_epoch = len(X_train) // batch_size

    for epoch in range(n_epochs):
        with tqdm.trange(batches_per_epoch, unit="batch", mininterval=0, disable=True) as bar:
            bar.set_description(f"Epoch {epoch}")
            for i in bar:
                start = i * batch_size
                X_batch = X_train[start:start+batch_size]
                y_batch = y_train[start:start+batch_size]
                # forward pass
                y_pred = model(X_batch)
                loss = loss_fn(y_pred, y_batch)
                # backward pass
                optimizer.zero_grad()
                loss.backward()
                # update weights
                optimizer.step()
                # print progress
                acc = (y_pred.round() == y_batch).float().mean()
                bar.set_postfix(
                    loss=float(loss),
                    acc=float(acc)
                )
    # evaluate accuracy at end of training
    y_pred = model(X_test)
    acc = (y_pred.round() == y_test).float().mean()
    return float(acc)

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
Más información sobre [`StratifiedKFold`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html).

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
Más información sobre [`KFold`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html).

- Scikit-learn ofrece múltiples funciones de validación cruzada k-fold.

- Proceso:
  - Una parte se usa como conjunto de test.

  - Las otras cuatro partes se combinan como conjunto de entrenamiento.

  - Se repite cinco veces, cambiando la parte de test.

- En cada iteración:
  - Se llama a la función `model_train()`.

  - Se obtiene la puntuación de accuracy.

  - Se guarda la puntuación en una lista.

- Al final, se calcula la media y la desviación estándar de las puntuaciones.

- El objeto `kfold` devuelve índices, no los datos divididos.

- Los índices se usan para extraer los conjuntos de entrenamiento y test durante la ejecución de `model_train()`.

Este enfoque proporciona una evaluación más robusta y representativa del rendimiento del modelo.

In [8]:
from sklearn.model_selection import StratifiedKFold

kfold = StratifiedKFold(n_splits=5, shuffle=True)
cv_scores = []
for train, test in kfold.split(X, y.flatten()):  # devuelve ÍNDICES, no datos
    acc = model_train(X[train], y[train], X[test], y[test])
    print("Accuracy: %.2f" % acc)
    cv_scores.append(acc)
print("Accuracy Total: %.2f%% (+/- %.2f%%)" % (np.mean(cv_scores)*100, np.std(cv_scores)*100))

Accuracy: 0.67


Accuracy: 0.64


Accuracy: 0.65


Accuracy: 0.62


Accuracy: 0.58
Accuracy Total: 63.01% (+/- 3.17%)


<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<div style="text-align: right"> <font size=6><i class="fa fa-coffee" aria-hidden="true" style="color:#004D7F"></i> </font></div>